# Lab 4 — Modele de recunoaștere a vorbirii (română)
**Scop:** model ASR pentru materiale audio/video din tezaurul național (lecții EducațieOnline.md, clasa a 4-a, Matematică).

## 1. Strategia de lucru
| Etapă | Ce facem | Modul |
|---|---|---|
| 1 | Colectare: lecții YouTube (educatieonline.md) + folder Google Drive | `asr.collect` |
| 2 | Data Card (surse, statistici, drepturi, limitări) | `asr.datacard` |
| 3 | Preprocesare: video→WAV 16 kHz, VAD, segmente 1–20 s, normalizare text, pseudo-etichete | `asr.segment`, `asr.labelers`, `asr.text_norm` |
| 4 | Set train/dev/test **pe vorbitor** + validare manuală 10% din test | `asr.splits`, `asr.validate` |
| 5 | Arhitectură: Wav2Vec2 (XLSR-53) + cap CTC | `asr.train` |
| 6 | Antrenare (fine-tuning, early stopping) | `asr.train` |
| 7 | Evaluare: WER, CER, erori, RTF | `asr.evaluate` |

## 2. Problema și soluțiile posibile
* **Problema:** nu avem transcrieri umane pentru lecții. Soluții: (a) transcriere manuală — corectă dar scumpă; (b) **pseudo-etichete** de la un model existent (RobinASR) + corectarea manuală a unui subset — alegerea noastră; (c) model multilingv gata făcut (Whisper) fără antrenare — bun ca *referință*, dar nu e „modelul nostru".
* **Modele:** DeepSpeech/Kaldi (RobinASR) vs. Wav2Vec2-CTC (self-supervised, performant cu puține date, ales) vs. Whisper (seq2seq, mare, lent).
* **Risc metodologic:** dacă etichetele vin de la Robin, un WER calculat față de ele măsoară *acordul cu Robin*, nu acuratețea. De aceea **doar subsetul validat manual** dă WER real, iar împărțirea train/test se face pe vorbitor, ca să nu existe scurgeri de voce.

> **Rulare:** local (`pip install -r requirements.txt` + ffmpeg) sau Colab cu GPU (încărcați proiectul și rulați `%cd Lab4`). Antrenarea XLSR-53 are nevoie de GPU (≥16 GB). Pentru un test rapid pe CPU: `ASR_SMOKE=1`.

In [ ]:
import os, sys, json, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "asr").exists():          # notebook rulat din alt director
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SMOKE = os.getenv("ASR_SMOKE") == "1"    # test rapid pe date sintetice, fără rețea/GPU
from asr.config import Config, SOURCES
from asr.segment import read_jsonl
cfg = Config().make_dirs()
if SMOKE:
    import tempfile
    cfg = Config(base_dir=tempfile.mkdtemp(prefix="asr_nb_"), vad="energy", label_workers=2,
                 validation_min_segments=5, validation_fraction=0.5).make_dirs()
print("Proiect:", cfg.base_dir, "| mod:", "SMOKE (sintetic)" if SMOKE else "date reale")
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "indisponibil (CPU)")

## Pasul 1 — Colectarea setului de date
SRC-01: linkurile YouTube de pe educatieonline.md (47 lecții, lista inclusă ca rezervă) · SRC-02: folderul Google Drive din enunț. Se descarcă doar pista audio. Dacă YouTube cere autentificare, puneți `cookies.txt` sau copiați manual fișierele în `asr_project/videos/educatieonline/` (nume `SRC01_<id>.mp4`).

In [ ]:
from asr import collect
if SMOKE:
    from asr.synth import make_corpus
    truth = make_corpus(cfg, n_rec=10, utt_per_rec=18)
    inventory = collect.build_inventory(cfg); collect.save_inventory(cfg, inventory)
else:
    inventory = collect.collect(cfg, sources=("SRC-01", "SRC-02"))   # LIMIT=5 pentru un test: limit=5
import pandas as pd
if not inventory:
    raise RuntimeError(
        "Nu am găsit niciun fișier audio/video în asr_project/videos/. Opțiuni: (1) descărcați manual folderul "
        "Google Drive din enunț în asr_project/videos/gdrive/ și/sau lecțiile în asr_project/videos/educatieonline/ "
        "(nume SRC01_<id>.mp4), (2) puneți orice MP4/WAV în asr_project/videos/manual/, apoi rulați din nou "
        "celula (sau: collect.save_inventory(cfg, collect.build_inventory(cfg))). Diagnostic YouTube: "
        "python run_pipeline.py check-sources")
df = pd.DataFrame(inventory)
display(df.groupby("source").agg(fisiere=("recording_id", "count"), ore=("duration_s", lambda x: round(x.sum()/3600, 2)),
                                  vorbitori=("speaker", "nunique")))
df.head()

## Pasul 2 — Data Card (versiunea inițială)
Se regenerează automat după fiecare etapă; toate cifrele provin din fișierele reale.

In [ ]:
from asr import datacard
card = datacard.write_data_card(cfg)
print(open(cfg.data_card_md, encoding="utf-8").read()[:1800])

## Pasul 3 — Preprocesare
**3a.** video → WAV mono 16 kHz (ffmpeg). **3b.** VAD (Silero) → segmente 1–20 s; pauzele <0,6 s se unesc; segmentele identice (md5) se elimină.

In [ ]:
from asr import segment
inventory = segment.extract_all(cfg, inventory)
segments = segment.segment_all(cfg, inventory)
import numpy as np
d = np.array([s["duration"] for s in segments])
print(f"{len(segments)} segmente · {d.sum()/3600:.2f} h · durată medie {d.mean():.1f}s (min {d.min():.1f}, max {d.max():.1f})")

**3c. Transcrieri (pseudo-etichete).** Serverul RobinASR primește `multipart/form-data` cu câmpul `file` și răspunde `{"status":"OK","transcription":...}`. URL-ul din enunț (`relate.racai.ro/...?path=robin/asr`) este interfața web; pentru API rulați Robin local: `docker run -p 8888:8888 racai/robinasr:cpu` și setați `ROBIN_API_URL=http://localhost:8888/transcribe`. Alternative: `HFLabeler("openai/whisper-small", cfg)` sau transcrieri proprii în CSV (`MappingLabeler`).
Textul este normalizat (litere mici, ş→ș, cifre și simboluri → litere) pentru vocabularul modelului.

In [ ]:
from asr import labelers
if SMOKE:
    from asr.synth import oracle_text
    labeler = labelers.MappingLabeler(lambda s: oracle_text(truth, s), name="oracle-smoke")
else:
    labeler = labelers.RobinLabeler(cfg.robin_url, cfg.robin_timeout)
    # alternativă: labeler = labelers.HFLabeler("openai/whisper-small", cfg)
stats = labelers.label_segments(cfg, segments, labeler, save_path=cfg.manifests_dir / "segments.jsonl")
stats

In [ ]:
from asr.text_norm import normalize_text
for t in ["Adunăm 12 + 8 = 20.", "Ştiţi că 25 000 de lei?", "50% din 200"]:
    print(f"{t!r:35} → {normalize_text(t)!r}")

## Pasul 4 — Set de antrenare / dezvoltare / test
Împărțire **pe vorbitor** (profesor): segmentele vecine din aceeași lecție sunt aproape duplicate; amestecarea lor între train și test ar umfla artificial rezultatele. Din test se aleg 10% pentru validare manuală.

In [ ]:
from asr import splits
sp = splits.make_splits(cfg, segments)
pd.DataFrame({k: {"segmente": len(v), "ore": round(sum(s["duration"] for s in v)/3600, 2),
                  "vorbitori": len({s["speaker"] for s in v})} for k, v in sp.items()}).T

### Validare manuală (10% din test)
Două variante: **(A)** interfața web — `python run_pipeline.py serve`, tab „✅ Validare Manuală”; **(B)** foaie CSV pentru Excel (celula de mai jos): completați `corectat`, sau `actiune=sterge`; lăsat gol = confirmat. Apoi importați și aplicați.
În modul SMOKE „omul” este simulat de oracol.

In [ ]:
from asr import validate
sheet = validate.export_sheet(cfg)           # → manifests/validation_sheet.csv
if SMOKE:
    for s in sp["test"]:
        if s.get("val_selected"):
            validate.record_decision(cfg, s["id"], oracle_text(truth, s), "confirmed")
else:
    print("➡ Completați", sheet, "(sau folosiți tab-ul Validare din interfață), apoi rulați celula următoare.")

In [ ]:
# După ce ați completat CSV-ul:  validate.import_sheet(cfg, sheet)
counts = validate.apply_validation(cfg)
counts

## Data Card (după pașii 1–4)

In [ ]:
card = datacard.write_data_card(cfg)
print(open(cfg.data_card_md, encoding="utf-8").read())

## Pasul 5 — Arhitectura modelului
**Wav2Vec2-XLSR-53 + cap CTC.** Forma de undă brută → encoder CNN (7 straturi, înghețat) → 24 blocuri Transformer (preantrenate auto-supravegheat pe 53 de limbi) → proiecție liniară peste vocabularul românesc (a–z, ă â î ș ț, spațiu, `[UNK]`, `[PAD]`) → pierdere CTC. Decodare: argmax + colapsarea repetițiilor. În SMOKE se folosește o variantă minusculă, inițializată aleator.

In [ ]:
from asr.dataset import build_vocab
from asr.train import load_model
vocab = build_vocab()
print("Vocabular:", len(vocab), "simboluri:", list(vocab))
model = load_model(cfg, len(vocab), vocab["[PAD]"], smoke=SMOKE)
tot = sum(p.numel() for p in model.parameters())/1e6
print(f"Model: {cfg.base_model if not SMOKE else 'tiny (smoke)'} · {tot:.1f}M parametri · straturi Transformer: {model.config.num_hidden_layers}")
del model

## Pasul 6 — Antrenarea
Fine-tuning cu AdamW, LR 1e-4 cu warm-up, gradient checkpointing, fp16 pe GPU, early stopping pe WER-ul din dev (patience 5), grupare pe lungime. Hiperparametrii sunt în `asr/config.py`.

In [ ]:
from asr.train import train
summary = train(cfg, smoke=SMOKE, max_steps=450 if SMOKE else None)
{k: v for k, v in summary.items() if k not in ("log_history", "hyperparameters")}

In [ ]:
import matplotlib.pyplot as plt
h = summary["log_history"]
tr = [(x["step"], x["loss"]) for x in h if "loss" in x]
ev = [(x["step"], x["eval_wer"]) for x in h if "eval_wer" in x]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(*zip(*tr)); ax[0].set_title("Loss antrenare"); ax[0].set_xlabel("pas")
if ev: ax[1].plot(*zip(*ev), marker="o"); ax[1].set_title("WER pe dev"); ax[1].set_xlabel("pas")
plt.tight_layout(); plt.show()

## Pasul 7 — Evaluarea
* **WER/CER pe subsetul validat manual** = acuratețea reală (referință corectată de om).
* WER pe tot testul = acord cu pseudo-etichetele (informativ).
* **„Profesor” (Robin) pe subsetul validat** = cât de bune erau pseudo-etichetele; arată dacă studentul îl depășește pe profesor.
* RTF = timp inferență / durată audio.

In [ ]:
from asr.evaluate import evaluate_split
res = evaluate_split(cfg, "test")
f = lambda x: "n/a" if x is None else f"{x:.1%}"
print(f"Validat manual : WER {f(res['validated']['wer'])} · CER {f(res['validated']['cer'])} ({res['validated']['n']} segmente)")
print(f"Tot testul     : WER {f(res['overall']['wer'])} · CER {f(res['overall']['cer'])} ({res['overall']['n']} segmente)")
if 'teacher_on_validated' in res: print(f"Profesor/Robin : WER {f(res['teacher_on_validated']['wer'])} pe același subset")
print("RTF:", res["rtf"], "| dispozitiv:", res["device"])
if res.get("warning"): print("⚠", res["warning"])
pd.DataFrame(res["top_confusions"]).head(10)

In [ ]:
for e in res["worst_examples"][:5]:
    print(e["id"]); print("  REF :", e["ref"]); print("  PRED:", e["hyp"])

## Inferență pe un fișier nou (inclusiv lecție lungă) și limba rusă

In [ ]:
from asr.inference import Transcriber
from asr.audio_io import load_audio
tr = Transcriber(str(cfg.final_dir), vad="energy" if SMOKE else "silero")
sample = sorted(cfg.audio_dir.glob("*.wav"))[0]
out = tr.transcribe(load_audio(sample))
print(f"{out['duration']:.0f}s audio în {out['processing_time']:.1f}s (RTF {out['rtf']})")
for s in out["segments"][:5]: print(f"[{s['start']:6.1f}–{s['end']:6.1f}] {s['text']}")

In [ ]:
# Rusă: model HuggingFace (necesită rețea). DeepSpeech-ul din enunț (GeorgeFedoseev) cere Python ≤3.9 și
# TensorFlow vechi; varianta practică este un model Wav2Vec2 rus, evaluat cu aceleași funcții.
if not SMOKE:
    ru = Transcriber(cfg.ru_model, vad="silero")
    # print(ru.transcribe(load_audio("exemplu_ru.wav"))["text"])
    print("Model rus încărcat:", cfg.ru_model)
else:
    print("(omis în SMOKE)")

## Server și interfață web
```
python run_pipeline.py serve          # → http://localhost:8000  (Transcriere · Validare · Istoric · Setări)
```
Swagger: `/docs`. Motoare: model local, RobinASR (prin server), rusă (HF).

In [ ]:
card = datacard.write_data_card(cfg)       # Data Card final, cu rezultatele evaluării
print("Gata. Fișiere în:", cfg.base_dir)